In [5]:
import os
import json
from typing import Dict

import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "./models/llama-3.1-8b-instruct"
TRAIN_JSONL_PATH = "train.jsonl"
OUTPUT_DIR = "./lora/adapters/resume_scorer_v1"
MAX_SEQ_LENGTH = 2048

# 1) 이 두 줄이 아주 중요: 라이브러리 차원에서 '오프라인 모드' 강제
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

def format_example(example: Dict) -> Dict:
    text = example["input"] + example["output"]
    return {"text": text}

def main():
    # --------------------------
    # 1. 데이터셋 로드
    # --------------------------
    if not os.path.exists(TRAIN_JSONL_PATH):
        raise FileNotFoundError(f"{TRAIN_JSONL_PATH} 를 찾을 수 없습니다.")

    raw_datasets: DatasetDict = load_dataset(
        "json",
        data_files={"train": TRAIN_JSONL_PATH},
    )

    ds = raw_datasets["train"]
    if len(ds) < 2:
        # 샘플이 1개 이하일 때: 그냥 전부 train으로 쓰고 eval은 없음
        train_ds = ds.map(format_example)
        eval_ds = None
    else:
        # 샘플이 2개 이상일 때만 train/test 나누기
        split = ds.train_test_split(test_size=0.1, seed=42)
        train_ds = split["train"]
        eval_ds = split["test"]

    # # train / eval 나누기 (단순 9:1 split)
    # raw_datasets = raw_datasets["train"].train_test_split(test_size=0.1, seed=42)
    # train_ds = raw_datasets["train"].map(format_example)
    # eval_ds = raw_datasets["test"].map(format_example)

    print("len(train_dataset) =", len(train_ds))

    # --------------------------
    # 2. 토크나이저 & 모델 로드 (4bit QLoRA)
    # --------------------------
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print(f"모델 로드 중... ({MODEL_NAME})")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        use_fast=True,
        local_files_only=True,   # 인터넷 절대 안 씀
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        local_files_only=True,   # 인터넷 절대 안 씀
    )

    # --------------------------
    # 3. LoRA 설정
    # --------------------------
    # target_modules 는 모델 구조에 따라 달라질 수 있음 (Llama 계열 기준 예시)
    peft_config = LoraConfig(
        r=4,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    # --------------------------
    # 4. 학습 설정
    # --------------------------
    sft_config = SFTConfig(
        output_dir=OUTPUT_DIR,

        # ===== 로깅 / 진행바 세팅 =====
        logging_strategy="steps",   # "no" / "epoch" / "steps"
        save_strategy="steps",
        disable_tqdm=False,         # tqdm progress bar 켜기
        report_to="none",           # wandb 안 쓸 거면 이렇게
        
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,   # 효과적인 batch_size ≈ 8
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=10,
        save_steps=200,
        # evaluation_strategy="steps",   # 필요하면 사용 (버전에 따라 "no"/"steps"/"epoch")
        eval_steps=200,
        save_total_limit=3,
        # bf16=True,
        fp16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        max_grad_norm=1.0,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
    
        # ✅ SFT 전용 옵션들
        dataset_text_field="text",       # 데이터셋에서 텍스트가 들어있는 컬럼 이름
        max_length=MAX_SEQ_LENGTH,
        packing=False,                   # 여러 샘플을 한 시퀀스로 패킹할지 여부
    )

    # --------------------------
    # 5. SFTTrainer 생성
    # --------------------------
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        peft_config=peft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        args=sft_config,
    )

    dl = trainer.get_train_dataloader()
    batch = next(iter(dl))
    print({k: v.shape for k, v in batch.items()})


    # --------------------------
    # 6. 학습 실행
    # --------------------------
    print("학습 시작!")
    trainer.train()

    # --------------------------
    # 7. LoRA 어댑터 + 토크나이저 저장
    # --------------------------
    print("모델 저장 중...")
    trainer.model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"저장 완료: {OUTPUT_DIR}")
    

if __name__ == "__main__":
    main()


len(train_dataset) = 1
모델 로드 중... (./models/llama-3.1-8b-instruct)


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [08:55<00:00, 133.99s/it]


{'input_ids': torch.Size([1, 2048]), 'labels': torch.Size([1, 2048]), 'attention_mask': torch.Size([1, 2048])}


C:\Portpolio\yonsei\HrAutoFlow\server\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
